# 04. XGBoost 모델 학습

## 목표
조성 기반 feature → 밴드갭 예측 회귀 모델 학습 + 성능 평가 + feature importance 추출.

feature importance가 3단계 도메인 해석의 원재료가 된다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

df = pd.read_csv('../data/featurized_highk.csv')
print('로드:', df.shape)

feature_cols = [c for c in df.columns if c not in ['material_id', 'formula', 'band_gap']]
X = df[feature_cols]
y = df['band_gap']
print(f'features: {len(feature_cols)}개 / 타겟: band_gap')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)
print('학습 완료')

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'MAE : {mae:.3f} eV')
print(f'R²  : {r2:.3f}')

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.4, s=15, color='steelblue')
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', linewidth=1)
plt.xlabel('Actual Band Gap (eV)')
plt.ylabel('Predicted Band Gap (eV)')
plt.title(f'XGBoost  MAE={mae:.3f} eV  R²={r2:.3f}')
plt.tight_layout()
plt.savefig('../data/prediction_scatter.png', dpi=150)
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols)
top20 = importance.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
top20[::-1].plot(kind='barh', color='steelblue')
plt.xlabel('Feature Importance (gain)')
plt.title('Top 20 Features — XGBoost Band Gap Prediction')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150)
plt.show()

print('\nTop 20 Feature Importance:')
print(top20.round(4))

In [ ]:
results = df.loc[X_test.index, ['formula', 'band_gap']].copy()
results['predicted'] = y_pred
results['error'] = abs(results['band_gap'] - results['predicted'])

print('=== 예측 오류 Top 10 ===')
print(results.nlargest(10, 'error')[['formula', 'band_gap', 'predicted', 'error']].to_string(index=False))

model.save_model('../data/xgb_bandgap_model.json')
print('\n모델 저장 완료')